In [1]:
import boto3
import pandas as pd
from io import BytesIO

BUCKET_NAME = "kltn-customer-analytics-2026"
FILE_KEY = "bronze/online_retail_II.xlsx"

s3 = boto3.client("s3")

response = s3.get_object(
    Bucket=BUCKET_NAME,
    Key=FILE_KEY
)

excel_data = BytesIO(response["Body"].read())

df = pd.read_excel(
    excel_data,
    sheet_name=None,
    engine="openpyxl"
)

df = pd.concat(df.values(), ignore_index=True)

df["InvoiceDate"] = pd.to_datetime(df["InvoiceDate"])

print("Kích thước dữ liệu:", df.shape)

Kích thước dữ liệu: (1067371, 8)


In [2]:
df_clean = df.copy()

cleaning_log = []

def log_step(step, before, after):
    cleaning_log.append({
        "Step": step,
        "Rows Before": before,
        "Rows Removed": before - after,
        "Rows After": after
    })

print("Dữ liệu ban đầu:", len(df_clean))

Dữ liệu ban đầu: 1067371


In [3]:
before = len(df_clean)

cancelled = (
    df_clean["Invoice"]
    .astype("string")
    .str.strip()
    .str.upper()
    .str.startswith("C", na=False)
)

df_clean = df_clean.loc[~cancelled].copy()

log_step(
    "Remove cancelled invoices",
    before,
    len(df_clean)
)

print("Số dòng còn lại:", len(df_clean))

Số dòng còn lại: 1047877


In [4]:
before = len(df_clean)

df_clean = df_clean.dropna(
    subset=["Customer ID"]
).copy()

log_step(
    "Remove missing Customer ID",
    before,
    len(df_clean)
)

print("Số dòng còn lại:", len(df_clean))

Số dòng còn lại: 805620


In [5]:
before = len(df_clean)

valid_transactions = (
    (df_clean["Quantity"] > 0) &
    (df_clean["Price"] > 0)
)

df_clean = df_clean.loc[
    valid_transactions
].copy()

log_step(
    "Remove non-positive Quantity or Price",
    before,
    len(df_clean)
)

print("Số dòng còn lại:", len(df_clean))

Số dòng còn lại: 805549


In [6]:
duplicate_count = df_clean.duplicated().sum()

print("Số dòng trùng còn lại:", duplicate_count)

print("\nMẫu dòng trùng:")
print(
    df_clean.loc[
        df_clean.duplicated(keep=False)
    ].head(10).to_string()
)

Số dòng trùng còn lại: 26124

Mẫu dòng trùng:
    Invoice StockCode                        Description  Quantity         InvoiceDate  Price  Customer ID         Country
362  489517     21913     VINTAGE SEASIDE JIGSAW PUZZLES         1 2009-12-01 11:34:00   3.75      16329.0  United Kingdom
363  489517     21912           VINTAGE SNAKES & LADDERS         1 2009-12-01 11:34:00   3.75      16329.0  United Kingdom
365  489517     21821   GLITTER STAR GARLAND WITH BELLS          1 2009-12-01 11:34:00   3.75      16329.0  United Kingdom
367  489517     22319  HAIRCLIPS FORTIES FABRIC ASSORTED        12 2009-12-01 11:34:00   0.65      16329.0  United Kingdom
368  489517     22130   PARTY CONE CHRISTMAS DECORATION          6 2009-12-01 11:34:00   0.85      16329.0  United Kingdom
371  489517     21912           VINTAGE SNAKES & LADDERS         1 2009-12-01 11:34:00   3.75      16329.0  United Kingdom
379  489517     21491    SET OF THREE VINTAGE GIFT WRAPS         1 2009-12-01 11:34:00   1.95

In [7]:
cleaning_report = pd.DataFrame(cleaning_log)

print("\nBÁO CÁO LÀM SẠCH DỮ LIỆU")
print(cleaning_report.to_string(index=False))

print("\nDữ liệu sau các bước lọc ban đầu:")
print("Số dòng:", len(df_clean))
print("Số khách hàng:", df_clean["Customer ID"].nunique())
print("Số dòng trùng:", df_clean.duplicated().sum())


BÁO CÁO LÀM SẠCH DỮ LIỆU
                                 Step  Rows Before  Rows Removed  Rows After
            Remove cancelled invoices      1067371         19494     1047877
           Remove missing Customer ID      1047877        242257      805620
Remove non-positive Quantity or Price       805620            71      805549

Dữ liệu sau các bước lọc ban đầu:
Số dòng: 805549
Số khách hàng: 5878
Số dòng trùng: 26124


In [8]:
duplicates = df_clean.duplicated()

duplicate_rows = df_clean.loc[duplicates].copy()

duplicate_rows["LineTotal"] = (
    duplicate_rows["Quantity"] * duplicate_rows["Price"]
)

print("Số dòng trùng:", len(duplicate_rows))

print(
    "Số hóa đơn có dòng trùng:",
    duplicate_rows["Invoice"].nunique()
)

print(
    "Số khách hàng có dòng trùng:",
    duplicate_rows["Customer ID"].nunique()
)

print(
    "Tổng giá trị các dòng trùng (GBP):",
    round(duplicate_rows["LineTotal"].sum(), 2)
)

print("\n10 dòng trùng mẫu:")
print(duplicate_rows.head(10).to_string(index=False))

print("\nPhân bố Quantity của các dòng trùng:")
print(
    duplicate_rows["Quantity"]
    .describe()
    .to_string()
)

Số dòng trùng: 26124
Số hóa đơn có dòng trùng: 4967
Số khách hàng có dòng trùng: 1893
Tổng giá trị các dòng trùng (GBP): 368624.91

10 dòng trùng mẫu:
Invoice StockCode                       Description  Quantity         InvoiceDate  Price  Customer ID        Country  LineTotal
 489517     21912          VINTAGE SNAKES & LADDERS         1 2009-12-01 11:34:00   3.75      16329.0 United Kingdom       3.75
 489517     22130  PARTY CONE CHRISTMAS DECORATION          6 2009-12-01 11:34:00   0.85      16329.0 United Kingdom       5.10
 489517     22319 HAIRCLIPS FORTIES FABRIC ASSORTED        12 2009-12-01 11:34:00   0.65      16329.0 United Kingdom       7.80
 489517     21913    VINTAGE SEASIDE JIGSAW PUZZLES         1 2009-12-01 11:34:00   3.75      16329.0 United Kingdom       3.75
 489517     21821  GLITTER STAR GARLAND WITH BELLS          1 2009-12-01 11:34:00   3.75      16329.0 United Kingdom       3.75
 489517    84951A   S/4 PISTACHIO LOVEBIRD COASTERS         1 2009-12-01 11:34:00

In [10]:
before = len(df_clean)

df_clean = df_clean.drop_duplicates().copy()

log_step(
    "Remove exact duplicate rows",
    before,
    len(df_clean)
)

print("Số dòng bị loại:", before - len(df_clean))
print("Số dòng còn lại:", len(df_clean))

Số dòng bị loại: 26124
Số dòng còn lại: 779425


In [11]:
# Chuyển Customer ID sang kiểu số nguyên nullable
df_clean["Customer ID"] = (
    df_clean["Customer ID"].astype("Int64")
)

# Tính giá trị từng dòng
df_clean["LineTotal"] = (
    df_clean["Quantity"] * df_clean["Price"]
)

print("Số dòng:", len(df_clean))
print("Số khách hàng:", df_clean["Customer ID"].nunique())
print("Số hóa đơn:", df_clean["Invoice"].nunique())

print("Customer ID thiếu:", df_clean["Customer ID"].isna().sum())
print("Dòng trùng:", df_clean.duplicated().sum())
print("Quantity <= 0:", (df_clean["Quantity"] <= 0).sum())
print("Price <= 0:", (df_clean["Price"] <= 0).sum())

print("Tổng LineTotal (GBP):", round(df_clean["LineTotal"].sum(), 2))

Số dòng: 779425
Số khách hàng: 5878
Số hóa đơn: 36969
Customer ID thiếu: 0
Dòng trùng: 0
Quantity <= 0: 0
Price <= 0: 0
Tổng LineTotal (GBP): 17374804.27


In [13]:
cleaning_report = pd.DataFrame(cleaning_log)

print(cleaning_report.to_string(index=False))

print("\nSố dòng ban đầu:", len(df))
print("Số dòng cuối cùng:", len(df_clean))
print(
    "Tỷ lệ giữ lại:",
    round(len(df_clean) / len(df) * 100, 2), "%"
)

                                 Step  Rows Before  Rows Removed  Rows After
            Remove cancelled invoices      1067371         19494     1047877
           Remove missing Customer ID      1047877        242257      805620
Remove non-positive Quantity or Price       805620            71      805549
          Remove exact duplicate rows       805549         26124      779425

Số dòng ban đầu: 1067371
Số dòng cuối cùng: 779425
Tỷ lệ giữ lại: 73.02 %


In [1]:
import pandas as pd

df_clean = pd.read_csv(
    "../output/online_retail_clean_backup.csv",
    parse_dates=["InvoiceDate"]
)

print(df_clean.shape)
print(round(df_clean["LineTotal"].sum(), 2))

(779425, 9)
17374804.27


In [15]:
df_clean.to_csv(
    "../output/online_retail_clean_backup.csv",
    index=False
)